# ACL Tear Detection — V8.2 MRNet Dataset + MRNet Architecture

**Key change from V8.1:** Uses the **official MRNet dataset** (1,250 patients) instead of the
smaller Kaggle MRI dataset (~736 patients). This should significantly reduce overfitting.

**Dataset:**
- **MRNet Stanford** — 1,130 train + 120 valid = 1,250 patients (sagittal plane)
- **Priyank Saxena** — 466 patients (external validation, never trained on)

**Architecture:** Same as V8.1 — MRNet-style max-pool across slices + EfficientNet-B0

**Settings carried from V8.1:**
- ImageNet normalization (required for EfficientNet)
- CrossEntropyLoss with class weights
- AdamW, lr=5e-5, weight_decay=0.01
- shuffle=True (no WeightedRandomSampler)
- Fully fine-tuned backbone
- Save best model on val_AUC

**What's different:**

| | V8.1 | V8.2 |
|---|---|---|
| Dataset | Kaggle MRI (~736) | **MRNet Stanford (1,250)** |
| Train patients | ~515 | **~1,000** |
| ACL tears (train) | ~135 | **~208** |
| Slice count | Fixed 10 | **Variable (17-51, use ALL)** |
| External val | Priyank | **Priyank** (same) |
| Slice selection | Source-aware fixed range | **Use all slices (like original MRNet)** |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
# MRNet sagittal data (converted from .npy to .npz)
MRNET_DATA_DIR = '/content/drive/MyDrive/dataset/mrnet_sagittal'
# Priyank data for external validation
PRIYANK_DATA_DIR = '/content/drive/MyDrive/dataset/combined'

BATCH_SIZE = 1           # MRNet uses batch_size=1 (variable slice count per volume)
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5
RANDOM_SEED = 42

# MRNet uses ALL slices — no fixed slice range
MAX_SLICES = 40          # Pad/truncate to this for batching

# Priyank slice range (for external validation only)
SLICE_RANGE_PRIYANK = (33, 43)

DROPOUT = 0.5
WEIGHT_DECAY = 0.01
PATIENCE = 10
N_FOLDS = 5

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load MRNet Metadata
# ============================================================
mrnet_path = Path(MRNET_DATA_DIR)
mrnet_df = pd.read_csv(mrnet_path / 'metadata.csv')
mrnet_df['label_binary'] = mrnet_df['label'].astype(int)
mrnet_df['label_name_binary'] = mrnet_df['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"MRNet Dataset: {len(mrnet_df)} patients")
print(mrnet_df['label_name_binary'].value_counts())
print(f"\nOriginal split:")
print(pd.crosstab(mrnet_df['split'], mrnet_df['label_name_binary'], margins=True))
print(f"\nSlice count: min={mrnet_df['num_slices'].min()}, max={mrnet_df['num_slices'].max()}, mean={mrnet_df['num_slices'].mean():.1f}")

# Load Priyank metadata for external validation
priyank_path = Path(PRIYANK_DATA_DIR)
all_metadata = pd.read_csv(priyank_path / 'metadata.csv')
priyank_df = all_metadata[all_metadata['source'] == 'Priyank_Saxena'].reset_index(drop=True)
priyank_df['label_binary'] = (priyank_df['label'] > 0).astype(int)
priyank_df['label_name_binary'] = priyank_df['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"\nPriyank (External Validation): {len(priyank_df)} patients")
print(priyank_df['label_name_binary'].value_counts())

In [ ]:
# ============================================================
# Dataset — MRNet volumes (uses ALL slices, like the original)
# ============================================================

class MRNetVolumeDataset(Dataset):
    """Dataset for MRNet .npz volumes. Uses ALL slices (no fixed range)."""

    def __init__(self, df, data_dir, max_slices=MAX_SLICES, augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.max_slices = max_slices
        self.augment = augment

        # ImageNet normalization for EfficientNet
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        # Verify files exist
        self.valid_indices = []
        for idx, row in self.df.iterrows():
            fpath = self.data_dir / row['filename']
            if fpath.exists():
                self.valid_indices.append(idx)

        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]

        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']  # (S, 256, 256), uint8

        # Use ALL slices (MRNet-style), normalize to [0,1]
        slices = volume.astype(np.float32) / 255.0

        # Pad or truncate to max_slices
        actual = slices.shape[0]
        if actual < self.max_slices:
            pad = np.zeros((self.max_slices - actual, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual > self.max_slices:
            # Center crop
            offset = (actual - self.max_slices) // 2
            slices = slices[offset:offset + self.max_slices]

        # Light augmentation
        if self.augment:
            if np.random.random() < 0.5:
                slices = slices[:, :, ::-1].copy()
            if np.random.random() < 0.5:
                import cv2
                angle = np.random.uniform(-15, 15)
                h, w = slices.shape[1], slices.shape[2]
                M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
                slices = np.stack([
                    cv2.warpAffine(s, M, (w, h), borderValue=0) for s in slices
                ])

        # 3-channel + ImageNet normalization
        slices_3ch = np.stack((slices,)*3, axis=1)  # (S, 3, H, W)
        slices_tensor = torch.FloatTensor(slices_3ch)
        slices_tensor = torch.stack([self.normalize(s) for s in slices_tensor])

        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]


class PriyankVolumeDataset(Dataset):
    """Dataset for Priyank .npz volumes (uses fixed slice range)."""

    def __init__(self, df, data_dir, slice_range=SLICE_RANGE_PRIYANK, max_slices=MAX_SLICES):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.slice_range = slice_range
        self.max_slices = max_slices

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        self.valid_indices = []
        for idx, row in self.df.iterrows():
            fpath = self.data_dir / row['filename']
            if fpath.exists():
                self.valid_indices.append(idx)

        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]

        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']

        start, end = self.slice_range
        start = min(start, volume.shape[0])
        end = min(end, volume.shape[0])
        slices = volume[start:end].astype(np.float32) / 255.0

        # Pad to max_slices
        actual = slices.shape[0]
        if actual < self.max_slices:
            pad = np.zeros((self.max_slices - actual, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual > self.max_slices:
            slices = slices[:self.max_slices]

        slices_3ch = np.stack((slices,)*3, axis=1)
        slices_tensor = torch.FloatTensor(slices_3ch)
        slices_tensor = torch.stack([self.normalize(s) for s in slices_tensor])

        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]

In [ ]:
# ============================================================
# Model — same as V8.1
# ============================================================

class ACLVolumeNet(nn.Module):
    """MRNet architecture with EfficientNet-B0 backbone."""
    def __init__(self, dropout=DROPOUT):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = backbone.features
        self.pool = backbone.avgpool
        self.in_features = backbone.classifier[1].in_features  # 1280

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.in_features, 2)
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"  Backbone: EfficientNet-B0, fully fine-tuned")
        print(f"  Classifier: Dropout({dropout}) -> Linear({self.in_features}, 2)")
        print(f"  Params: {trainable:,} trainable / {total:,} total")

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        features = self.features(x)
        features = self.pool(features)
        features = features.flatten(1)
        features = features.view(B, S, -1)
        pooled, _ = torch.max(features, dim=1)
        logits = self.classifier(pooled)
        return logits

In [ ]:
# ============================================================
# Training & Validation — same as V8.1
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    for volumes, labels, _ in tqdm(loader, desc='Training', leave=False):
        volumes = volumes.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(volumes)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for volumes, labels, _ in tqdm(loader, desc='Validating', leave=False):
            volumes = volumes.to(device)
            labels = labels.to(device)

            logits = model(volumes)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc, all_preds, all_labels

---
## Part A: Single-Split Training (MRNet → Priyank External Validation)

In [ ]:
# ============================================================
# Train/Val/Test Split — MRNet Data
# Option 1: Use MRNet's original train/valid split
# Option 2: Re-split all 1250 patients ourselves
# Using Option 2 for consistency with V8.1 (70/15/15)
# ============================================================
train_df, temp_df = train_test_split(
    mrnet_df, test_size=0.3, stratify=mrnet_df['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print("MRNet Dataset Split:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_name_binary'].value_counts().to_dict()
    print(f"  {name}: {len(df)} patients — {dist}")

print(f"\nExternal (Priyank): {len(priyank_df)} patients — {priyank_df['label_name_binary'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# Create Datasets & DataLoaders
# ============================================================
print("Creating datasets...")
print(f"  MAX_SLICES={MAX_SLICES} (pad/truncate)")
print()

print("Train (MRNet, augmented):")
train_dataset = MRNetVolumeDataset(train_df, MRNET_DATA_DIR, augment=True)
print("Val (MRNet):")
val_dataset = MRNetVolumeDataset(val_df, MRNET_DATA_DIR, augment=False)
print("Test (MRNet):")
test_dataset = MRNetVolumeDataset(test_df, MRNET_DATA_DIR, augment=False)
print("Priyank (external):")
priyank_dataset = PriyankVolumeDataset(priyank_df, PRIYANK_DATA_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
priyank_loader = DataLoader(priyank_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain: {len(train_dataset)} patients, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} patients")
print(f"Test:  {len(test_dataset)} patients")
print(f"Priyank: {len(priyank_dataset)} patients")

In [ ]:
# ============================================================
# Model, Loss, Optimizer
# ============================================================
print("Creating model...")
model = ACLVolumeNet()
model = model.to(device)

# Class weights
train_labels = train_dataset.get_labels()
n_total = len(train_labels)
n_pos = sum(train_labels)
n_neg = n_total - n_pos
w_normal = n_total / (2 * n_neg)
w_tear = n_total / (2 * n_pos)
class_weights = torch.FloatTensor([w_normal, w_tear]).to(device)
print(f"  Class counts: Normal={n_neg}, Tear={n_pos}")
print(f"  Class weights: Normal={w_normal:.3f}, Tear={w_tear:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print(f"\nSettings:")
print(f"  Loss:       CrossEntropyLoss (class-weighted)")
print(f"  Optimizer:  AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"  Scheduler:  ReduceLROnPlateau on val_AUC (factor=0.5, patience=3)")
print(f"  Early stop: patience={PATIENCE} on val_AUC")

In [ ]:
# ============================================================
# Training Loop — save on best val_AUC
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'train_auc': [],
           'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc = 0.0
best_val_loss = float('inf')
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v8.2.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE} on val_AUC)...\n")

for epoch in range(NUM_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']

    train_loss, train_acc, train_auc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_auc, val_preds, val_labels = validate(
        model, val_loader, criterion, device
    )

    scheduler.step(val_auc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    gap = train_acc - val_acc
    gap_warn = ' OVERFIT' if gap > 10 else ''

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  (lr={current_lr:.1e})")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%  AUC={train_auc:.4f}")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  AUC={val_auc:.4f}  (gap={gap:.1f}%){gap_warn}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> Saved best (AUC={val_auc:.4f}, acc={val_acc:.2f}%, loss={val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nBest val AUC: {best_val_auc:.4f}, val_loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(history['train_auc'], label='Train')
axes[2].plot(history['val_auc'], label='Val')
axes[2].set_title('AUC'); axes[2].legend(); axes[2].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[3].plot(gaps, color='red')
axes[3].axhline(y=10, color='orange', linestyle='--', label='10% threshold')
axes[3].set_title('Overfit Gap'); axes[3].legend(); axes[3].grid(True)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v8.2.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluate on MRNet Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))

test_loss, test_acc, test_auc, test_preds, test_labels = validate(
    model, test_loader, criterion, device
)

label_names = ['Normal', 'Tear']

print('=' * 60)
print('TEST RESULTS — V8.2 (MRNet Data)')
print('=' * 60)
print(f'Accuracy: {test_acc:.2f}%')
print(f'AUC: {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — MRNet Test (V8.2)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8.2_mrnet.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# External Validation — Priyank Saxena
# ============================================================
print('=' * 60)
print('EXTERNAL VALIDATION — Priyank Saxena')
print('=' * 60)

eval_criterion = nn.CrossEntropyLoss()
priyank_loss, priyank_acc, priyank_auc, priyank_preds, priyank_labels = validate(
    model, priyank_loader, eval_criterion, device
)

print(f'Accuracy: {priyank_acc:.2f}%')
print(f'AUC: {priyank_auc:.4f}')
print('\nClassification Report:')
print(classification_report(priyank_labels, priyank_preds, target_names=label_names, digits=3))

cm_p = confusion_matrix(priyank_labels, priyank_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_p, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — Priyank External (V8.2)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8.2_priyank.png', dpi=150)
plt.show()

print(f"\n{'='*60}")
print(f"V8.2 SUMMARY")
print(f"{'='*60}")
print(f"  MRNet Test: {test_acc:.1f}% acc, AUC={test_auc:.4f}")
print(f"  Priyank:    {priyank_acc:.1f}% acc, AUC={priyank_auc:.4f}")

---
## Part B: 5-Fold Stratified CV (MRNet) + Priyank External

In [ ]:
# ============================================================
# K-Fold CV
# ============================================================
print(f"\n{'='*60}")
print(f"{N_FOLDS}-FOLD STRATIFIED CV (MRNet) + Priyank External")
print(f"V8.2")
print(f"{'='*60}\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
all_fold_preds = []
all_fold_labels = []
all_fold_priyank_preds = []
all_fold_priyank_labels = []

print("Priyank dataset:")
priyank_cv_dataset = PriyankVolumeDataset(priyank_df, PRIYANK_DATA_DIR)
priyank_cv_loader = DataLoader(
    priyank_cv_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

for fold, (train_idx, val_idx) in enumerate(skf.split(mrnet_df, mrnet_df['label_binary'])):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    fold_train_df = mrnet_df.iloc[train_idx]
    fold_val_df = mrnet_df.iloc[val_idx]

    print(f"  Train: {len(fold_train_df)} — {fold_train_df['label_name_binary'].value_counts().to_dict()}")
    print(f"  Val:   {len(fold_val_df)} — {fold_val_df['label_name_binary'].value_counts().to_dict()}")

    print("  Train dataset:")
    fold_train_ds = MRNetVolumeDataset(fold_train_df, MRNET_DATA_DIR, augment=True)
    print("  Val dataset:")
    fold_val_ds = MRNetVolumeDataset(fold_val_df, MRNET_DATA_DIR, augment=False)

    fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    fold_val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print("  Creating model...")
    fold_model = ACLVolumeNet()
    fold_model = fold_model.to(device)

    fold_labels = fold_train_ds.get_labels()
    fold_n_pos = sum(fold_labels)
    fold_n_neg = len(fold_labels) - fold_n_pos
    fold_w = torch.FloatTensor([
        len(fold_labels) / (2 * fold_n_neg),
        len(fold_labels) / (2 * fold_n_pos)
    ]).to(device)
    fold_criterion = nn.CrossEntropyLoss(weight=fold_w)

    fold_optimizer = optim.AdamW(fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    fold_scheduler = optim.lr_scheduler.ReduceLROnPlateau(fold_optimizer, mode='max', factor=0.5, patience=3)

    best_fold_auc = 0.0
    fold_patience = 0
    best_fold_state = None

    for epoch in range(NUM_EPOCHS):
        t_loss, t_acc, t_auc = train_epoch(fold_model, fold_train_loader, fold_criterion, fold_optimizer, device)
        v_loss, v_acc, v_auc, _, _ = validate(fold_model, fold_val_loader, fold_criterion, device)
        fold_scheduler.step(v_auc)

        if v_auc > best_fold_auc:
            best_fold_auc = v_auc
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} (best AUC: {best_fold_auc:.4f})")
                break

    fold_model.load_state_dict(best_fold_state)

    _, fold_acc, fold_auc, fold_preds, fold_labels_list = validate(
        fold_model, fold_val_loader, fold_criterion, device
    )
    print(f"\n  MRNet Val: acc={fold_acc:.2f}%, AUC={fold_auc:.4f}")
    print(classification_report(fold_labels_list, fold_preds, target_names=label_names, digits=3, zero_division=0))

    eval_crit = nn.CrossEntropyLoss()
    _, p_acc, p_auc, p_preds, p_labels = validate(fold_model, priyank_cv_loader, eval_crit, device)
    print(f"  Priyank: acc={p_acc:.2f}%, AUC={p_auc:.4f}")
    print(classification_report(p_labels, p_preds, target_names=label_names, digits=3, zero_division=0))

    fold_results.append({
        'fold': fold + 1, 'mri_acc': fold_acc, 'mri_auc': fold_auc,
        'priyank_acc': p_acc, 'priyank_auc': p_auc,
        'best_epoch': epoch + 1 - PATIENCE if fold_patience >= PATIENCE else epoch + 1,
    })

    all_fold_preds.extend(fold_preds)
    all_fold_labels.extend(fold_labels_list)
    all_fold_priyank_preds.extend(p_preds)
    all_fold_priyank_labels.extend(p_labels)

    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# K-Fold Summary
# ============================================================
print(f"\n{'='*60}")
print(f"K-FOLD CV SUMMARY (V8.2 — MRNet Dataset)")
print(f"{'='*60}")

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

print(f"\nMRNet:   {results_df['mri_acc'].mean():.2f}% +/- {results_df['mri_acc'].std():.2f}%  AUC={results_df['mri_auc'].mean():.4f}")
print(f"Priyank: {results_df['priyank_acc'].mean():.2f}% +/- {results_df['priyank_acc'].std():.2f}%  AUC={results_df['priyank_auc'].mean():.4f}")

print(f"\nMRNet Patient-Level (all folds):")
print(classification_report(all_fold_labels, all_fold_preds, target_names=label_names, digits=3, zero_division=0))

print(f"Priyank Patient-Level (all folds):")
print(classification_report(all_fold_priyank_labels, all_fold_priyank_preds, target_names=label_names, digits=3, zero_division=0))

In [ ]:
# ============================================================
# K-Fold Visualization
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

x = results_df['fold']
w = 0.3

axes[0].bar(x - w/2, results_df['mri_acc'], width=w, label='MRNet', color='steelblue')
axes[0].bar(x + w/2, results_df['priyank_acc'], width=w, label='Priyank', color='coral')
axes[0].set_title('Accuracy per Fold'); axes[0].legend(); axes[0].set_xticks(x); axes[0].grid(axis='y')

axes[1].bar(x - w/2, results_df['mri_auc'], width=w, label='MRNet', color='steelblue')
axes[1].bar(x + w/2, results_df['priyank_auc'], width=w, label='Priyank', color='coral')
axes[1].set_title('AUC per Fold'); axes[1].legend(); axes[1].set_xticks(x); axes[1].grid(axis='y')

cm_all = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_all, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[2])
axes[2].set_title(f'MRNet {N_FOLDS}-Fold CM')

cm_p_all = confusion_matrix(all_fold_priyank_labels, all_fold_priyank_preds)
sns.heatmap(cm_p_all, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[3])
axes[3].set_title(f'Priyank {N_FOLDS}-Fold CM')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/kfold_results_v8.2.png', dpi=150)
plt.show()